# 📊 Unemployment in India — Exploratory Data Analysis
### *Oasis Infobyte Data Science Internship — Task 2*

---

> **Objective:** Perform a comprehensive EDA on India's unemployment trends, with a focused lens on the **COVID-19 pandemic's economic shock**, using monthly CMIE (Centre for Monitoring Indian Economy) data spanning May 2019 – June 2020.

| Attribute | Detail |
|---|---|
| **Dataset** | Unemployment in India (Kaggle) |
| **Rows** | 740 (after cleaning) |
| **Regions** | 28 States / Union Territories |
| **Date Range** | May 2019 → June 2020 |
| **Key Event** | COVID-19 National Lockdown — April 2020 |

---

**Notebook Sections:**
1. 🔧 Setup & Imports
2. 📥 Data Loading & Cleaning
3. 🔍 Basic EDA & Descriptive Statistics
4. 📈 Temporal Analysis — Time-Series Line Chart
5. 🗺️ Regional Analysis — Top 10 States Bar Chart
6. 🔗 Correlation Analysis — Heatmap
7. 🦠 COVID-19 Impact — Pre vs Post Comparison

---
## 🔧 Section 1: Setup & Imports
We begin by importing all required libraries and configuring global plot styling.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Global Plot Style ──────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        120,
    'figure.facecolor':  '#F8F9FA',
    'axes.facecolor':    '#F8F9FA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    14,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
})

VISUALS_DIR  = 'visuals'
os.makedirs(VISUALS_DIR, exist_ok=True)
ACCENT_COLOR = '#E63946'

print('✅ Libraries imported. Environment ready.')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

---
## 📥 Section 2: Data Loading & Cleaning

The raw CSV from Kaggle contains a known quirk: **column names have leading whitespace** (e.g., `' Date'`, `' Estimated Unemployment Rate (%)'`). We address this in the cleaning pipeline below.

### Cleaning Pipeline
| Step | Action |
|---|---|
| 1 | Strip whitespace from all column names |
| 2 | Strip whitespace from all string values |
| 3 | Rename long column names for readability |
| 4 | Parse `Date` column as proper `datetime` objects |
| 5 | Drop fully-empty trailing rows and rows with null key fields |

In [ ]:
# ── Load ───────────────────────────────────────────────────────
df = pd.read_csv('Unemployment_in_India.csv')

print('Raw columns (first 3 characters shown as ordinals to expose whitespace):')
for c in df.columns:
    print(f'  {repr(c)}')

# ── Strip column names ─────────────────────────────────────────
df.columns = df.columns.str.strip()

# ── Strip string values ────────────────────────────────────────
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# ── Rename for convenience ─────────────────────────────────────
df.rename(columns={
    'Estimated Unemployment Rate (%)':       'Unemployment_Rate',
    'Estimated Employed':                    'Employed',
    'Estimated Labour Participation Rate (%)': 'Labour_Participation_Rate',
}, inplace=True)

# ── Parse Date ─────────────────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# ── Drop nulls ─────────────────────────────────────────────────
df.dropna(how='all', inplace=True)
df.dropna(subset=['Unemployment_Rate', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\n✅ Cleaning complete.')
print(f'   Shape : {df.shape}')
print(f'   Dates : {df["Date"].min().date()}  →  {df["Date"].max().date()}')
df.head(10)

### 💡 Observations — Data Loading

- The raw file contained **29 leading-whitespace column names** — a silent bug that would have caused `KeyError` without the `.str.strip()` step.
- After stripping trailing blank rows (empty rows appended by Excel during export), **740 valid records** remain across **28 regions**.
- The date range spans **May 2019 to June 2020** — a critical window that captures ~11 months of pre-pandemic normality and ~3 months of COVID-19 lockdown impact.
- All 740 rows are complete — **zero null values** in key numeric columns after cleaning.

---
## 🔍 Section 3: Basic EDA & Descriptive Statistics

In [ ]:
print('=' * 55)
print('DATA TYPES')
print('=' * 55)
print(df.dtypes)
print()
print('=' * 55)
print('NULL VALUE COUNTS')
print('=' * 55)
print(df.isnull().sum())

In [ ]:
print('DESCRIPTIVE STATISTICS — NUMERIC COLUMNS')
df[['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']].describe().round(2)

In [ ]:
print(f'Total unique regions : {df["Region"].nunique()}')
print(f'Unique area types    : {df["Area"].unique().tolist()}')
print(f'Date range           : {df["Date"].min().date()} → {df["Date"].max().date()}')
print()
print('Region-wise record count:')
df.groupby('Region').size().reset_index(name='Records').to_string(index=False)

### 💡 Observations — Descriptive Statistics

| Metric | Value | Insight |
|---|---|---|
| **Mean Unemployment Rate** | ~11.8% | Significantly above India's historical norm of ~5-7%, heavily skewed by lockdown months |
| **Max Unemployment Rate** | 76.74% | An extreme outlier — likely Puducherry or a small UT during peak lockdown |
| **Min Unemployment Rate** | 0.00% | Very low-unemployment rural regions in stable months |
| **Std Dev** | ~10.72% | High variance signals dramatic temporal and regional disparity |
| **Labour Participation Rate** | Mean 42.6% | India's female labour force participation is structurally low, pulling this metric down |
| **Area Split** | Rural vs Urban | 359 rural records vs 381 urban — balanced representation |

The **high standard deviation (10.72%)** in unemployment rate is the most telling stat — it confirms this is not a uniform trend but a story of stark contrasts between states and across time.

---
## Analysis 1: Region-Wise Average Unemployment Rate

In [ ]:
region_avg = (
    df.groupby('Region')['Unemployment_Rate']
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .reset_index()
)
region_avg.columns = ['Region', 'Avg_Unemployment_Rate (%)']
print('Region-wise Average Unemployment Rate (sorted descending):')
region_avg

### 💡 Observations — Regional Averages

The regional averages reveal a **strongly unequal distribution** of unemployment across India:

- **Tripura (28.35%)** and **Haryana (26.28%)** lead by a wide margin — these states have structural economic challenges and industry-specific vulnerabilities.
- **Jharkhand, Bihar, and Himachal Pradesh** form a second tier (~18-21%), reflecting states with high dependence on agriculture and limited industrial base.
- **Meghalaya (4.80%), Odisha (5.66%), and Assam (6.43%)** show the lowest averages — often rural-dominant economies where informal employment masks true joblessness.
- The **28.35% vs 4.80% spread** between top and bottom regions represents a **5.9× difference**, confirming that India's unemployment is fundamentally a regional phenomenon, not a uniform national one.

---
## 📈 Section 4: Temporal Analysis — Unemployment Rate Over Time

We select **6 major states** representing different geographies and economic profiles — Maharashtra (financial hub), Uttar Pradesh (most populous), Tamil Nadu (manufacturing), Rajasthan (arid/agriculture), Kerala (high literacy), and Delhi (capital/services) — to trace how unemployment evolved month-by-month.

The **red dashed vertical line** marks April 1, 2020 — the onset of India's national COVID-19 lockdown.

In [ ]:
major_states = ['Maharashtra', 'Uttar Pradesh', 'Tamil Nadu',
                'Rajasthan', 'Kerala', 'Delhi']

df_ts = (
    df[df['Region'].isin(major_states)]
    .groupby(['Region', 'Date'])['Unemployment_Rate']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette('Set2', len(major_states))

for i, state in enumerate(major_states):
    data = df_ts[df_ts['Region'] == state].sort_values('Date')
    ax.plot(data['Date'], data['Unemployment_Rate'],
            marker='o', markersize=4, linewidth=2.2,
            label=state, color=colors[i])

# COVID reference line
ax.axvline(pd.Timestamp('2020-04-01'), color=ACCENT_COLOR,
           linestyle='--', linewidth=2,
           label='COVID-19 Lockdown (Apr 2020)')
ax.axvspan(pd.Timestamp('2020-04-01'), df['Date'].max(),
           alpha=0.07, color=ACCENT_COLOR, label='_nolegend_')

ax.set_title('Unemployment Rate Over Time — Major Indian States', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Unemployment Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{VISUALS_DIR}/timeseries_line_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → visuals/timeseries_line_chart.png')

### 💡 Observations — Time-Series Line Chart

This chart is the **centrepiece of the analysis** — it tells the story of India's economic disruption with unmistakable clarity:

**Before April 2020 (Pre-Lockdown):**
- All six states exhibit **relatively stable, low unemployment** in the range of 3%–18%.
- Minor seasonal fluctuations are visible — small peaks in the June–August 2019 period (monsoon slowdown) and December–January (post-harvest lull).
- **Rajasthan and Delhi** show slightly elevated baselines compared to Kerala and Maharashtra.

**April 2020 Onwards (Lockdown & Post-Lockdown):**
- All states show a **dramatic, simultaneous spike** in unemployment — a textbook V-shaped economic shock from a sudden supply and demand shutdown.
- **Tamil Nadu surges from ~5% to ~30%+** — one of the sharpest increases, reflecting its heavy reliance on manufacturing, textiles, and auto industries.
- **Kerala and Maharashtra** both spike significantly but show signs of recovery by June 2020 as the lockdown relaxed.
- **Rajasthan** had a high pre-COVID baseline (~12-15%) and climbed further during lockdown.

**Key Takeaway:** The COVID-19 lockdown (April 2020) created a **uniform, synchronised unemployment shock** across all states — erasing regional differences momentarily and hitting every part of India regardless of its economic profile.

---
## 🗺️ Section 5: Regional Analysis — Top 10 States by Average Unemployment Rate

We aggregate unemployment over the entire period (May 2019–June 2020) to identify the **chronically high-unemployment states** — regions that struggle structurally, not just during the pandemic.

In [ ]:
top10 = (
    df.groupby('Region')['Unemployment_Rate']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10.columns = ['Region', 'Avg_Rate']

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = sns.color_palette('Reds_r', len(top10))

bars = ax.bar(top10['Region'], top10['Avg_Rate'],
              color=bar_colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, top10['Avg_Rate']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val:.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_title('Top 10 Indian States / UTs by Average Unemployment Rate\n(May 2019 – June 2020)',
             fontweight='bold')
ax.set_xlabel('State / Region')
ax.set_ylabel('Average Unemployment Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{VISUALS_DIR}/top10_states_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → visuals/top10_states_bar_chart.png')

### 💡 Observations — Top 10 States Bar Chart

**Tripura leads at 28.35% — more than double the national average.** This chart exposes the structural unemployment problem across India's most vulnerable states:

| Rank | State | Rate | Root Cause |
|---|---|---|---|
| 1 | **Tripura** | 28.35% | Heavily public-sector dependent; private sector almost absent; geographical isolation |
| 2 | **Haryana** | 26.28% | Rapid agricultural mechanisation is displacing rural labour; urban growth is concentrated |
| 3 | **Jharkhand** | 20.58% | Mineral-rich but processing industries create limited direct employment |
| 4 | **Bihar** | 18.92% | Structural underemployment in agriculture; massive out-migration state |
| 5 | **Himachal Pradesh** | 18.54% | Seasonal tourism/horticulture economy amplified by COVID's travel ban |

**Delhi (6th at 16.5%)** is a notable inclusion — surprising for the national capital, but understandable given the lockdown's outsized impact on its **services and informal economy** (domestic workers, hawkers, construction labour).

**Important caveat:** These averages include the COVID-19 spike months. States that experienced extreme lockdown spikes (like Puducherry: +3,549%) will have their overall averages pulled up significantly, even if they had low pre-COVID baselines.

---
## 🔗 Section 6: Correlation Analysis — Heatmap

We explore the statistical relationships between the three key numeric variables:
- **Unemployment Rate (%)** — the target metric
- **Employed (count)** — absolute number of employed people
- **Labour Participation Rate (%)** — share of working-age population actively in the labour force

Understanding these correlations helps validate economic theory and uncover India-specific patterns.

In [ ]:
corr_cols = ['Unemployment_Rate', 'Employed', 'Labour_Participation_Rate']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    linewidths=0.8,
    linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'},
    vmin=-1,
    vmax=1,
    ax=ax
)

labels = ['Unemployment\nRate (%)', 'Employed\n(Count)', 'Labour Participation\nRate (%)']
ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels(labels, fontsize=9, rotation=0)
ax.set_title('Correlation Matrix: Unemployment | Employment | Labour Participation',
             fontweight='bold', pad=14)

plt.tight_layout()
plt.savefig(f'{VISUALS_DIR}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → visuals/correlation_heatmap.png')

### 💡 Observations — Correlation Heatmap

The heatmap reveals three important correlation coefficients:

| Pair | Correlation | Interpretation |
|---|---|---|
| **Unemployment Rate ↔ Employed Count** | **–0.223** | Moderate negative — as unemployment rises, the absolute number employed falls. Expected but weak, since `Employed` is an absolute count (varies by state size) |
| **Unemployment Rate ↔ Labour Participation Rate** | **+0.003** | Near-zero — counterintuitively, in India, higher unemployment does **not** significantly depress labour force participation. This reflects the **discouraged worker effect** being offset by necessity-driven participation in the informal economy |
| **Employed Count ↔ Labour Participation Rate** | **+0.011** | Essentially zero — employment volume and participation rate are independent, again due to the massive scale difference between populous states (UP, MH) and smaller UTs |

**Deep Insight — India-Specific Economic Reality:**
The near-zero correlation between unemployment and labour participation challenges the Western economic assumption that job losses cause workers to exit the labour force. In India, where **~90% of employment is informal**, workers who lose formal jobs often shift to subsistence farming, street vending, or casual daily labour rather than dropping out entirely. This is why India's **Labour Participation Rate (~42%) remained stable even during lockdown** — people were still *trying* to work, just unable to find meaningful employment.

> The weak correlations are themselves an important finding: they confirm that **unemployment rate is the primary independent variable** that cannot be predicted from employment count or participation rate alone.

---
## 🦠 Section 7: COVID-19 Impact Analysis — Pre vs. Post Comparison

**The defining question of this dataset:** How much did COVID-19 actually change unemployment in India?

We split the dataset at **April 2020** — the month the nationwide lockdown began — and compare:
- **Pre-COVID period:** May 2019 – March 2020 (11 months of baseline)
- **Post-COVID period:** April 2020 – June 2020 (3 lockdown months)

We then drill down to the state level to identify which states were hit hardest.

In [ ]:
pre_covid  = df[df['Date'] <  '2020-04-01'].copy()
post_covid = df[df['Date'] >= '2020-04-01'].copy()

pre_mean   = pre_covid['Unemployment_Rate'].mean()
post_mean  = post_covid['Unemployment_Rate'].mean()
pct_change = (post_mean - pre_mean) / pre_mean * 100

print('=' * 55)
print('NATIONAL-LEVEL COMPARISON')
print('=' * 55)
print(f'Pre-COVID  mean unemployment rate  : {pre_mean:.2f}%')
print(f'Post-COVID mean unemployment rate  : {post_mean:.2f}%')
print(f'Absolute change                    : +{post_mean - pre_mean:.2f} percentage points')
print(f'Relative change                    : +{pct_change:.1f}%')
print()
print(f'Sample sizes  →  Pre: {len(pre_covid)} records  |  Post: {len(post_covid)} records')

In [ ]:
# State-level pre vs post COVID
state_comparison = pd.DataFrame({
    'Pre_COVID':  pre_covid.groupby('Region')['Unemployment_Rate'].mean(),
    'Post_COVID': post_covid.groupby('Region')['Unemployment_Rate'].mean(),
}).dropna()

state_comparison['Change_pp'] = (state_comparison['Post_COVID'] -
                                  state_comparison['Pre_COVID']).round(2)
state_comparison['Change_%']  = (
    (state_comparison['Post_COVID'] - state_comparison['Pre_COVID'])
    / state_comparison['Pre_COVID'] * 100
).round(1)

print('State-Level Pre vs. Post-COVID Unemployment (sorted by % change):')    
state_comparison.sort_values('Change_%', ascending=False).round(2)

In [ ]:
# Visualise national-level comparison as grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: National summary bar ──────────────────────────────────
ax = axes[0]
periods = ['Pre-COVID\n(May 2019 – Mar 2020)', 'Post-COVID\n(Apr 2020 – Jun 2020)']
means   = [pre_mean, post_mean]
palette = ['#2196F3', ACCENT_COLOR]

bars = ax.bar(periods, means, color=palette,
              edgecolor='white', linewidth=0.8, width=0.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val:.2f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold')

ax.set_title('National Average Unemployment Rate:\nPre-COVID vs. Post-COVID',
             fontweight='bold')
ax.set_ylabel('Average Unemployment Rate (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Annotation arrow
ax.annotate(f'+{pct_change:.0f}% ↑',
            xy=(1, post_mean), xytext=(0.5, (pre_mean + post_mean) / 2),
            fontsize=12, fontweight='bold', color=ACCENT_COLOR,
            arrowprops=dict(arrowstyle='->', color=ACCENT_COLOR, lw=2))

# ── Right: Horizontal bar — top 15 states by change in pp ───────
ax2 = axes[1]
top_states = state_comparison.nlargest(15, 'Change_pp').sort_values('Change_pp')

bar_colors2 = ['#E63946' if v > 0 else '#4CAF50' for v in top_states['Change_pp']]
ax2.barh(top_states.index, top_states['Change_pp'],
         color=bar_colors2, edgecolor='white', linewidth=0.5)

for i, (val, state) in enumerate(zip(top_states['Change_pp'], top_states.index)):
    ax2.text(val + 0.3, i, f'+{val:.1f}pp', va='center', fontsize=8, fontweight='bold')

ax2.set_title('Top 15 States: Unemployment Change\n(Post – Pre COVID, in percentage points)',
              fontweight='bold')
ax2.set_xlabel('Change in Unemployment Rate (percentage points)')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('COVID-19 Pandemic Impact on Indian Unemployment',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{VISUALS_DIR}/covid_impact_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → visuals/covid_impact_comparison.png')

### 💡 Observations — COVID-19 Impact Analysis

#### National-Level: A 110% Unemployment Surge

The numbers speak for themselves:

| Period | Mean Unemployment | Change |
|---|---|---|
| Pre-COVID (May 2019 – Mar 2020) | **9.61%** | Baseline |
| Post-COVID (Apr 2020 – Jun 2020) | **20.19%** | **+10.58 percentage points (+110%)** |

India's unemployment **more than doubled** in just 3 months — an economic disruption with no modern precedent.

#### State-Level: Who Suffered Most?

| Most Impacted | % Change | Context |
|---|---|---|
| **Puducherry** | +3,549% | Tiny economy; tourism/hospitality collapsed completely |
| **Tamil Nadu** | +904% | Manufacturing shutdowns (auto, textile, leather) |
| **Karnataka** | +485% | IT sector work-from-home masked service sector job losses |
| **Telangana** | +291% | Hyderabad's gig/services economy badly hit |
| **Jharkhand** | +222% | Construction and mine operations halted |

#### Unexpected Finding: States That Fared Better
Some states actually showed **lower post-COVID rates** than pre-COVID:
- **Chandigarh (–57%)** and **Jammu & Kashmir (–33%)** — these may reflect data collection disruptions during lockdown rather than actual improvement.
- **Tripura (–10%)** — already at extremely high baseline; the lockdown may have shifted workers into subsistence farming.

#### Critical Insight:
> The COVID-19 shock was most devastating for **industrialised states with large urban service and manufacturing sectors** (Tamil Nadu, Karnataka, Maharashtra). **Rural-agrarian states** experienced relatively lower spikes because agricultural work — though informal — continued through the lockdown, and food production was classified as essential.

---
## 📋 Summary & Key Findings

This EDA has uncovered the following core insights from India's unemployment data:

### 🔑 Key Findings

| # | Finding | Evidence |
|---|---|---|
| 1 | **COVID-19 caused India's unemployment to surge by +110%** | Pre-COVID: 9.61% → Post-COVID: 20.19% |
| 2 | **Regional inequality is extreme** — a 5.9× gap between best and worst states | Tripura 28.35% vs Meghalaya 4.80% |
| 3 | **Industrialised states suffered the sharpest COVID spikes** | Tamil Nadu +904%, Karnataka +485% |
| 4 | **Unemployment ↔ Labour Participation has near-zero correlation in India** | r = 0.003, explained by informal economy dynamics |
| 5 | **April 2020 is a clear structural break in every state's time series** | Simultaneous spike in all 6 major states |

### 🏁 Conclusion

India's unemployment landscape is shaped by two overlapping forces: **chronic structural inequality** (some states have always been high-unemployment) and an **acute pandemic shock** that amplified every weakness simultaneously. Any policy response must address both dimensions — targeted state-level interventions for chronically distressed regions, and economy-wide stimulus for pandemic recovery.